## SomeCens package tutorial 🌎📊

Package code in available at [SomeCens's github repository](github.com/jimenaRL/SoMeCens).   

#### In this tutorial we will use SomeCens package in order to:

- Create a Demograph object representing a country and its geographic units (the country's geographical subdivisions) 🌐
- Load sociodemographic (age and gender) information for units 👶👧👵👦🧔👴
- Localize user in units 📌
- Create choropleth maps to show sociodemographic and localisation data 🗺️
- Export tables with localized user and units sociodemographic data 📊

We will use data already formatted in the correct way for using them with the Demograph class.
You can check the [scripts](https://github.com/jimenaRL/SoMeCens/tree/0fddfc2ff01611bf9f21af2a36b2e648b1e2bbd2/scripts) provided in the repository for formatting data for EU countries 🇪🇺, Chile 🇨🇱 and USA 🇺🇸.  

In [ ]:
country = 'chile'
unitspath = "data/chile/chile_geounits_census_2024.csv"
agedistpath = "data/chile/chile_age_distribution_census_2024.csv"
genderdistpath = "data/chile/chile_gender_distribution_census_2024.csv"
usersdatapath = "data/chile/chile_metadata_2023.csv"

#### Load main package and configurations variables for some countries

In [ ]:
import os, csv, time, yaml, json
from string import Template

In [ ]:
from somecens import DemoGraph
from somecens.tools import matchUsersLocations, getCountryAliases, getUnitsAliases, getOtherCountriesNames

In [ ]:
from somecens.chile.conf import CHILEAGECATS, CHILEGENDERCATS
from somecens.us.conf import USAGECATS, USGENDERCATS
from somecens.nuts.conf import NUTS3AGECATS, NUTS3GENDERCATS

#### 1. Demograph creation

In order to create a **Demograph object** representing a country (here Chile), we need to provide an iterable of dictionaries with the information of each geaographical units. The dictionaries must be of the form:

    {
        'code': 'FRJ24',
        'level': '3',
        'label': 'Gers',
        'parent_code': 'FRJ2'
    }

_There must be one and only one unit representing the highest country level with code "0" and a empty parent_code._

Here we load them from a flat csv file containing these informations.

In [ ]:
!xan head data/chile/chile_geounits_census_2024.csv | xan v

In [ ]:
with open(unitspath, "r", encoding="utf-8") as f:
    geoUnits = [r for r in csv.DictReader(f)]

geoUnits[:3]

We must also state which will be the age and gender categories of the country.

In [ ]:
print(f"CHILE AGE CATEGORIES:\n\t{CHILEAGECATS}")

print(f"\nCHILE GENDER CATEGORIES:\n\t{CHILEGENDERCATS}")

In [ ]:
demo = DemoGraph(
    demography=geoUnits,
    genderCats=CHILEGENDERCATS, 
    ageCats=CHILEAGECATS
)

We can show the tree structure of the demograph.

In [ ]:
demo.showGeoUnits(max_level=1)

#### 2. Load sociodemographic data

We can now load to the Demograph age and gender data for the geophical units.

2.1 For _**gender distributions**_ we need to provide an iterable of dictionaries containing each the code of an unit and the values for each of the gender categories. They are of the form:

    {
        'code': 'FRJ24',
        'total': '18480432',
        'male': '8967033',
        'female': '9513399'
    }

we get them from a flat csv file containing these informations.

In [ ]:
!xan head data/chile/chile_gender_distribution_census_2024.csv | xan v

In [ ]:
with open(genderdistpath, "r") as f:
    genderDistribution = [r for r in csv.DictReader(f)]

genderDistribution[:3]

In [ ]:
demo.setGenderDistributions(genderDistribution)

In [ ]:
demo.showGeoUnits(max_level=0)

2.2 For _**age distributions**_ we need to provide an iterable of dictionaries containing each the code of an unit and the value for each one of the age categories. They are of the form:


In [ ]:
!xan head -l 25 data/chile/chile_age_distribution_census_2024.csv | xan v

In [ ]:
with open(agedistpath, "r") as f:
    ageDistribution = [r for r in csv.DictReader(f)]

ageDistribution[:3]

In [ ]:
demo.setAgeDistributions(ageDistribution, isFlat=True)

In [ ]:
demo.showGeoUnits(max_level=0)

##### 3.1 We load first the metadata for users from Chile

#### 3. Users location match

Now that our demograph is created and loaded with demographic data, we will use it to mach the localisations declared by users to the geographical units.

We use the method called **matchUsersLocations** from the tools module of SomeCens.

In [ ]:
!xan head data/chile/chile_metadata_2023.csv | xan v 

In [ ]:
with open(usersdatapath, 'r') as f:
    metadata = [r for r in csv.reader(f)]
print(f"Locations file with {len(metadata)} entries loaded from {agedistpath}")
metadata[:3]

In [ ]:
STOPWORDS = "el|la|lo|les|las|los|de|del|en"

In [ ]:
match_kwargs = {
    "stopwords": STOPWORDS,
    "split_characters": ["-", "/", "|", ".", "'", "(", ")"],
    "search_index": [1],
    "has_headers": True,
}

In [ ]:
aliases = {demo.countryCode: getCountryAliases(country)}
aliases.update(getUnitsAliases(country))
aliases

In [ ]:
other_countries = getOtherCountriesNames(country)
banned_words = other_countries
list(banned_words)[:5]

We must also  informe the **matchUsersLocations** about the units and its labels

In [ ]:
locations = demo.getAllSubUnits()
locations

In [ ]:
start = time.time()
users_matched_locations = matchUsersLocations(
    locations=locations,
    data=metadata,
    aliases=aliases,
    banned_words=banned_words,
    verbose=False,
    **match_kwargs)

duration = time.time() - start
print(f"Whole matching {len(metadata)} users locations took {duration} seconds.")

In [ ]:
demo.setUsersLocations(users_matched_locations)

In [ ]:
demo.showGeoUnits(max_level=0)

#### 4. Exports

Now that the demograph has matched and store users is their geographical units, we export flat tables with  ...

In [ ]:
excel_export_path = "results/20251201/chile/chile_units_users_reports_nuts_2024_epo_2023.xlsx"
stats_export_pattern = "results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_${level}.csv"
users_export_path = "results/20251201/chile/localized_users_nuts_2024_epo_2023.csv"
full_users_export_path = "results/20251201/chile/localized_users_full_nuts_2024_epo_2023.csv"
units_export_path = "results/20251201/chile/units_nuts_2024.csv"


##### 4.1 Export matchs stats per level for cloropleths visualizations

In [ ]:
for level in range(demo.getDeepestLevel() + 1) :
    path = Template(stats_export_pattern).safe_substitute(level=level)
    demo.exportLocalizationsMatchesPerc(level, path, descendants=True, add_headers=False)

In [ ]:
!xan v results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_1.csv

##### 4.2 export localized users

In [ ]:
localizedUsers, localizedUsersColumns = demo.exportLocalizedUsers(users_export_path, full_path=full_users_export_path)

In [ ]:
! xan head results/20251201/chile/localized_users_nuts_2024_epo_2023.csv | xan v

In [ ]:
! xan head results/20251201/chile/localized_users_full_nuts_2024_epo_2023.csv | xan v

##### 4.3 export flatten units excel file to monitoring and debugging

In [ ]:
# 4.O 
unitReport, unitsColumns = demo.exportUnitsReport(units_export_path)


In [ ]:
import pandas as pd
with pd.ExcelWriter(excel_export_path) as writer:

    unitsColumns = [" ".join(c.split("_")) for c in unitsColumns]
    pd.DataFrame(data=unitReport, columns=unitsColumns) \
        .to_excel(writer, index=False, sheet_name=f"units stats")

    localizedUsersColumns = [" ".join(c.split("_")) for c in localizedUsersColumns]
    df = pd.DataFrame(data=localizedUsers, columns=localizedUsersColumns)
    df = df.sample(n=min(len(df), 10000), random_state=84)
    try:
        df.to_excel(writer, index=False, sheet_name=f"localized users")
    except:
        df['location'] = df['location'].apply(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) else x)
        df.to_excel(writer, index=False, sheet_name=f"localized users")